In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop         = fm.FontProperties(fname=font_path,      size=11)
font_prop_small   = fm.FontProperties(fname=font_path,      size=7)
font_prop_numbers = fm.FontProperties(fname=font_path,      size=9)
font_prop_title   = fm.FontProperties(fname=font_path_bold, size=14)

FIG_WIDTH = 16
CELL_BORDER_COLOR = 'white'

folders_filtered = ["Chronos-2", "Moirai-2", "Lag-Llama", "TimeGPT", "TimesFM", "TabPFN-TS"]

# ---------------------------------------------------------
# Location code -> abbreviation mapping (FluSight locations.csv, shared across challenges)
# ---------------------------------------------------------
locations_ref = pd.read_csv('../auxiliary-files/locations.csv', dtype={'location': str})
loc_to_abbr = dict(zip(locations_ref['location'], locations_ref['abbreviation']))
full_locations = locations_ref['abbreviation'].tolist()

## 1. Function to build correctly-normalized pivot tables (quantile correction) for a given challenge\n\n- `pivot_date`: normalized by `n_instances` (total rows in a single reference-date file, across all locations/horizons/quantiles).\n- `pivot_loc`: normalized by the total instance count **specific to each (model, location)**, summed across all reference dates — derived directly from the raw model-output files, since the pre-computed correction log alone does not contain the total number of predictions made.

In [ ]:
def build_quantile_correction_pivots(challenge):
    """
    Legge il log di correzione quantili pre-calcolato per un dato challenge (FluSight o CovidHub),
    e costruisce pivot_date/pivot_loc correttamente normalizzati leggendo anche i file grezzi
    di model-output (necessari per calcolare il vero totale di istanze per location).
    """
    df_q = pd.read_csv(f"../model-evaluation/correction-quantiles/correction_quantiles_log_{challenge}.csv")


    # mappa location -> abbreviation
    df_q['location'] = df_q['location'].astype(str).str.zfill(2)
    df_q['location_abbr'] = df_q['location'].map(loc_to_abbr)

    # ---------------------------------------------------------
    # Leggi i file grezzi di model-output per calcolare:
    # - n_instances_by_loc: totale istanze per (model, location), su tutte le reference_date
    # - n_instances: totale istanze per una singola reference_date (su tutte le location/horizon/quantili)
    # - full_date_range: tutte le reference_date effettivamente presenti per questo challenge
    # ---------------------------------------------------------
    all_rows_count = []
    all_dates = set()
    n_instances = None

    for folder in folders_filtered:
        files = [f for f in Path(f"../model-output/{challenge}/{folder}").resolve().iterdir()
                 if f.is_file() and f.suffix == '.csv'
                 and '2025-11-22' <= f.name[:10] <= '2026-05-23']  # evaluation window


        for file in files:
            df_raw = pd.read_csv(file)

            if n_instances is None:
                n_instances = df_raw.shape[0]  # totale istanze per una singola reference_date

            if 'reference_date' in df_raw.columns:
                all_dates.update(df_raw['reference_date'].astype(str).unique())

            loc_counts = df_raw.groupby('location').size().reset_index(name='n_total_loc')
            loc_counts['model'] = folder
            all_rows_count.append(loc_counts)

    loc_totals_df = pd.concat(all_rows_count, ignore_index=True)
    loc_totals_df['location'] = loc_totals_df['location'].astype(str).str.zfill(2)
    loc_totals_df['location_abbr'] = loc_totals_df['location'].map(loc_to_abbr)

    n_instances_by_loc = (
        loc_totals_df
        .groupby(['model', 'location_abbr'])['n_total_loc']
        .sum()
        .reset_index()
    )

    full_date_range = sorted(all_dates)

    # ---------------------------------------------------------
    # pivot_date: normalizza per n_instances (totale per reference_date)
    # ---------------------------------------------------------
    pivot_date = df_q.pivot_table(
        index='model', columns='reference_date', values='correction',
        aggfunc='size', fill_value=0
    )
    pivot_date = pivot_date.reindex(index=folders_filtered, columns=full_date_range, fill_value=0)
    pivot_date = pivot_date / n_instances * 100

    # ---------------------------------------------------------
    # pivot_loc: normalizza per il totale istanze SPECIFICO di ciascuna location e modello
    # ---------------------------------------------------------
    pivot_loc_counts = df_q.pivot_table(
        index='model', columns='location_abbr', values='correction',
        aggfunc='size', fill_value=0
    )
    pivot_loc_counts = pivot_loc_counts.reindex(index=folders_filtered, columns=full_locations, fill_value=0)

    pivot_loc_totals = n_instances_by_loc.pivot_table(
        index='model', columns='location_abbr', values='n_total_loc', fill_value=0
    )
    pivot_loc_totals = pivot_loc_totals.reindex(index=folders_filtered, columns=full_locations, fill_value=0)

    pivot_loc = (pivot_loc_counts / pivot_loc_totals.replace(0, np.nan) * 100)

    pivot_date = pivot_date.replace(0, np.nan)
    pivot_loc = pivot_loc.replace(0, np.nan)

    return pivot_date, pivot_loc, df_q, n_instances_by_loc

## 2. Build pivots for both challenges

In [ ]:
pivot_date_flu, pivot_loc_flu, df_q_flu, n_instances_by_loc_flu = build_quantile_correction_pivots('FluSight')
pivot_date_covid, pivot_loc_covid, df_q_covid, n_instances_by_loc_covid = build_quantile_correction_pivots('CovidHub')

pivot_date_flu

In [ ]:
pivot_loc_covid

## 3. Plot combined figure: 2x2 panels (A/B = FluSight, C/D = CovidHub)

In [ ]:
def style_heatmap_axis(ax, cbar_label):
    """Applica font_prop a tutti gli elementi testuali di una heatmap"""
    plt.setp(ax.get_xticklabels(), fontproperties=font_prop)
    plt.setp(ax.get_yticklabels(), fontproperties=font_prop)
    ax.set_xlabel(ax.get_xlabel(), fontproperties=font_prop)
    ax.set_ylabel(ax.get_ylabel(), fontproperties=font_prop)
    ax.set_title(ax.get_title(), fontproperties=font_prop_title, loc='left')

    cbar = ax.collections[0].colorbar
    cbar.set_label(cbar_label, fontproperties=font_prop)
    plt.setp(cbar.ax.get_yticklabels(), fontproperties=font_prop)


# ordina le location (colonne di pivot_loc) per frequenza totale decrescente, sommando FluSight+CovidHub,
# cosi' l'ordine delle colonne resta coerente tra i due pannelli B e D
col_order = (pivot_loc_flu.fillna(0).sum(axis=0) + pivot_loc_covid.fillna(0).sum(axis=0)).sort_values(ascending=False).index
pivot_loc_flu = pivot_loc_flu[col_order]
pivot_loc_covid = pivot_loc_covid[col_order]

# colormap personalizzata: bianco panna (0/NaN) -> rosso (max)
CREAM = '#F5EFE6'
reds_colors = plt.cm.Reds(np.linspace(0, 1, 256))
reds_colors[0] = mcolors.to_rgba(CREAM)
cmap_with_nan = mcolors.LinearSegmentedColormap.from_list('reds_cream', reds_colors)
cmap_with_nan.set_bad(color=CREAM)

fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, 10), dpi=300)

panels = [
    (axes[0, 0], pivot_date_flu,   'A. FluSight — percentage of quantiles correction by reference date', 45),
    (axes[0, 1], pivot_loc_flu,    'B. FluSight — percentage of quantiles correction by location',        90),
    (axes[1, 0], pivot_date_covid, 'C. CovidHub — percentage of quantiles correction by reference date',  45),
    (axes[1, 1], pivot_loc_covid,  'D. CovidHub — percentage of quantiles correction by location',         90),
]

for ax, pivot, title, rotation in panels:
    sns.heatmap(
        pivot, cmap=cmap_with_nan, linewidths=0.5, linecolor=CELL_BORDER_COLOR,
        cbar_kws={'label': 'Percentage of quantiles correction (%)'}, ax=ax,
        xticklabels=True, yticklabels=True
    )
    ax.set_xlabel(' ')
    ax.set_ylabel(' ')
    style_heatmap_axis(ax, 'Percentage of quantiles correction (%)')
    ax.tick_params(axis='x', rotation=rotation)
    ax.set_title(title, fontproperties=font_prop_title, loc='left', pad=10)

plt.tight_layout()
plt.savefig('../model-evaluation/figures/monothonicity-violation-quantiles_combined.png', dpi=300, bbox_inches='tight')
plt.savefig('../model-evaluation/figures/monothonicity-violation-quantiles_combined.svg', dpi=300, bbox_inches='tight')
plt.show()